In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import h5py
from scipy.fft import fftn, ifftn, fftfreq
import illustris_python as il
from matplotlib.colors import LogNorm
import seaborn as sns

In [ ]:
zs = [0.0,0.1,0.2,0.3,0.4,0.5,0.7,1.0,1.5,2.0,3.01,4.01,5.0,6.01,7.01,8.01,9.0,10.0]
snaps_ids = [99,91,84,78,72,67,59,50,40,33,25,21,17,13,11,8,6,4]

In [ ]:
norm_dens = []
voids = []
filaments = []
nodes = []
walls = []
dens = []

with h5py.File('/Users/users/roana/roana/BSc_Thesis/nexus_outputs.h5', 'r') as f:
    for red in zs:
        norm_dens.append(np.transpose(f[f'z_{red}']['normalized_density'][:], (1,2,0)))
        dens.append(np.transpose(f[f'z_{red}']['density'][:], (1,2,0)))
        voids.append(np.transpose(f[f'z_{red}']['voids'][:], (1,2,0)))
        filaments.append(np.transpose(f[f'z_{red}']['filaments'][:], (1,2,0)))
        nodes.append(np.transpose(f[f'z_{red}']['nodes'][:], (1,2,0)))
        walls.append(np.transpose(f[f'z_{red}']['walls'][:], (1,2,0)))
        print(f"Suessfully Loaded Stuff for z={red}")

n = len(norm_dens)
print(n)

In [ ]:
basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"

coords_0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['Coordinates'])/1000 #in cMpc/h
ids_0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['ParticleIDs'])
vels_0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['Velocities'])

L = 35.0 # cMpc/h
res = 128

dl  = L / res
print (dl)

In [ ]:
print("At redshift z=0...")

bool_filament = filaments[0].astype(bool)
bool_wall = walls[0].astype(bool)
bool_node = nodes[0].astype(bool)
bool_void = voids[0].astype(bool)

grid_x = np.floor(coords_0[:, 0] / dl).astype(int) % res
grid_y = np.floor(coords_0[:, 1] / dl).astype(int) % res
grid_z = np.floor(coords_0[:, 2] / dl).astype(int) % res

in_node_mask = bool_node[grid_x, grid_y, grid_z]

node_particle_ids = ids_0[in_node_mask]
print(f"Total DM particles: {len(ids_0)}")
print(f"Particles in nodes: {len(node_particle_ids)}")

in_filament_mask = bool_filament[grid_x, grid_y, grid_z]

filament_particle_ids = ids_0[in_filament_mask]
print(f"Particles in filaments: {len(filament_particle_ids)}")

in_wall_mask = bool_wall[grid_x, grid_y, grid_z]

wall_particle_ids = ids_0[in_wall_mask]
print(f"Particles in walls: {len(wall_particle_ids)}")

in_void_mask = bool_void[grid_x, grid_y, grid_z]

void_particle_ids = ids_0[in_void_mask]
print(f"Particles in voids: {len(void_particle_ids)}")

In [ ]:
match_mask = np.isin(ids_0, node_particle_ids)
node_vels_0 = vels_0[match_mask]
print("Recovered velocities for particles in nodes")

match_mask = np.isin(ids_0, filament_particle_ids)
filament_vels_0 = vels_0[match_mask]
print("Recovered velocities for particles in filaments")

match_mask = np.isin(ids_0, wall_particle_ids)
wall_vels_0 = vels_0[match_mask]
print("Recovered velocities for particles in walls")

match_mask = np.isin(ids_0, void_particle_ids)
void_vels_0 = vels_0[match_mask]
print("Recovered velocities for particles in voids")

In [ ]:
node_speeds_0 = np.sqrt(node_vels_0[:,0]**2 + node_vels_0[:,1]**2 + node_vels_0[:,2]**2)
filament_speeds_0 = np.sqrt(filament_vels_0[:,0]**2 + filament_vels_0[:,1]**2 + filament_vels_0[:,2]**2)
wall_speeds_0 = np.sqrt(wall_vels_0[:,0]**2 + wall_vels_0[:,1]**2 + wall_vels_0[:,2]**2)
void_speeds_0 = np.sqrt(void_vels_0[:,0]**2 + void_vels_0[:,1]**2 + void_vels_0[:,2]**2)

In [ ]:
plt.hist(node_speeds_0, bins="scott", color = "orange", density=True, histtype = "step",label = "Nodes")
plt.hist(filament_speeds_0, bins="scott", color = "g", density=True, histtype = "step",label = "Filaments")
plt.hist(wall_speeds_0, bins="scott", color = "r", density=True, histtype = "step",label = "Walls")
plt.hist(void_speeds_0, bins="scott", color = "b", density=True, histtype = "step",label = "Voids")
plt.xlabel(r"km $\sqrt{a}$ / s")
plt.ylabel("PDF")
#plt.xscale("log")
plt.grid()
plt.title("Speed distribution at z=0")
plt.legend();

In [ ]:
snap_z = 0 

particles_z = il.snapshot.loadSubset(basePath, snap_z, 'dm', ['Coordinates', 'ParticleIDs', 'Velocities'])
coords_z = particles_z['Coordinates']/1000
ids_z = particles_z['ParticleIDs']
vels_z = particles_z['Velocities']

match_mask = np.isin(ids_z, node_particle_ids)
node_vels_z = vels_z[match_mask]
print("Recovered velocities for particles in nodes")

match_mask = np.isin(ids_z, filament_particle_ids)
filament_vels_z = vels_z[match_mask]
print("Recovered velocities for particles in filaments")

match_mask = np.isin(ids_z, wall_particle_ids)
wall_vels_z = vels_z[match_mask]
print("Recovered velocities for particles in walls")

match_mask = np.isin(ids_z, void_particle_ids)
void_vels_z = vels_z[match_mask]
print("Recovered velocities for particles in voids")

In [ ]:
node_speeds_z = np.sqrt(node_vels_z[:,0]**2 + node_vels_z[:,1]**2 + node_vels_z[:,2]**2)
filament_speeds_z = np.sqrt(filament_vels_z[:,0]**2 + filament_vels_z[:,1]**2 + filament_vels_z[:,2]**2)
wall_speeds_z = np.sqrt(wall_vels_z[:,0]**2 + wall_vels_z[:,1]**2 + wall_vels_z[:,2]**2)
void_speeds_z = np.sqrt(void_vels_z[:,0]**2 + void_vels_z[:,1]**2 + void_vels_z[:,2]**2)

In [ ]:
plt.hist(node_speeds_z, bins="scott", color = "orange", density=True, histtype = "step",label = "Nodes")
plt.hist(filament_speeds_z, bins="scott", color = "g", density=True, histtype = "step",label = "Filaments")
plt.hist(wall_speeds_z, bins="scott", color = "r", density=True, histtype = "step",label = "Walls")
plt.hist(void_speeds_z, bins="scott", color = "b", density=True, histtype = "step",label = "Voids")
plt.xlabel(r"km $\sqrt{a}$ / s")
plt.ylabel("PDF")
plt.grid()
plt.title("Speed distribution at z=20.05")
plt.legend();

In [ ]:
# snap_initial = 0
# snap_final = 99

# ids_0 = il.snapshot.loadSubset(basePath, snap_initial, 'dm', fields = ['ParticleIDs'])
# vels_0 = il.snapshot.loadSubset(basePath, snap_initial, 'dm', fields = ['Velocities'])

# np.random.seed(42)
# subset_indices = np.random.choice(len(ids_0), size=10000, replace=False)

# ids_0_target = ids_0[subset_indices]

# vels_0_target_norms = np.linalg.norm(vels_0_target, axis=1)
# vels_0_target_hat = vels_0_target / vels_0_target_norms[:, np.newaxis]

# snaps = range(snap_initial, snap_final + 1)
# mean_alignments = []

# for i in snaps:
#     if (i % 10 == 0):
#         print(f"Working on snaps {i} to {i+9 }....")
#     ids = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['ParticleIDs'])
#     vels = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['Velocities'])

#     sort_idx = np.argsort(ids)
#     sorted_ids = ids[sort_idx]
    
#     match_indices = np.searchsorted(sorted_ids, ids_0_target)
#     original_indices = sort_idx[match_indices]

    
#     matched_vels = vels[original_indices]
    
#     matched_vels_norms = np.linalg.norm(matched_vels, axis=1)
#     matched_v_hat = matched_vels / matched_vels_norms[:, np.newaxis]
    
#     dot_products = np.sum(vels_0_target_hat * matched_v_hat, axis=1)
    
#     mean_alignments.append(np.mean(dot_products))

# print("Done!")


In [ ]:
ids_0 = il.snapshot.loadSubset(basePath, 0, 'dm', fields = ['ParticleIDs'])
vels_0 = il.snapshot.loadSubset(basePath, 0, 'dm', fields = ['Velocities'])

id_list = [ids_0, node_particle_ids, filament_particle_ids, wall_particle_ids, void_particle_ids]

mean_alignments = [[], [], [], [], []]
alignments_err_up = [[], [], [], [], []]
alignments_err_down = [[], [], [], [], []]
everything = [[], [], [], [], []]

vels_0_target_hat_list = []

snap_initial = 0
snap_final = 99
snaps = range(snap_initial, snap_final + 1)

np.random.seed(42)

id_list_new = []
vel_list_new = []

print("Computing preliminaries....")

sort_idx_0 = np.argsort(ids_0)
sorted_ids_0 = ids_0[sort_idx_0]

for k in range(5):

    target = id_list[k]

    subset_indices = np.random.choice(len(target), size=9000, replace=False)
    
    ids_0_target = target[subset_indices]

    match_indices_0 = np.searchsorted(sorted_ids_0, ids_0_target)
    original_indices_0 = sort_idx_0[match_indices_0]

    vels_0_target = vels_0[original_indices_0]
    
    vels_0_target_norms = np.linalg.norm(vels_0_target, axis=1)
    vels_0_target_hat = vels_0_target / vels_0_target_norms[:, np.newaxis]
    
    vels_0_target_hat_list.append(vels_0_target_hat)

    id_list_new.append(ids_0_target)
    vel_list_new.append(vels_0_target)

for i in snaps:

    if (i % 10 == 0):
        print(f"Working on snaps {i} to {i+9 }....")

    ids = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['ParticleIDs'])
    vels = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['Velocities'])

    sort_idx = np.argsort(ids)
    sorted_ids = ids[sort_idx]

    for k in range(5):

        match_indices = np.searchsorted(sorted_ids, id_list_new[k])
        original_indices = sort_idx[match_indices]

        matched_vels = vels[original_indices]
        matched_vels_norms = np.linalg.norm(matched_vels, axis=1)
        matched_v_hat = matched_vels / matched_vels_norms[:, np.newaxis]

        dot_products = np.sum(vels_0_target_hat_list[k] * matched_v_hat, axis=1)

        mean = np.mean(dot_products)
        higher = np.mean(dot_products[np.where(dot_products >= mean)]) - mean
        lower  = mean - np.mean(dot_products[np.where(dot_products <= mean)]) 

        everything[k].append(dot_products)
        mean_alignments[k].append(mean)
        alignments_err_up[k].append(higher)
        alignments_err_down[k].append(lower)


print("Done!")


In [ ]:
import numpy as np
import illustris_python as il # Assuming this is your import based on il.snapshot

ids_0 = il.snapshot.loadSubset(basePath, 0, 'dm', fields = ['ParticleIDs'])
vels_0 = il.snapshot.loadSubset(basePath, 0, 'dm', fields = ['Velocities'])

id_list = [ids_0, node_particle_ids, filament_particle_ids, wall_particle_ids, void_particle_ids]

# --- Existing lists for alignment ---
mean_alignments = [[], [], [], [], []]
alignments_err_up = [[], [], [], [], []]
alignments_err_down = [[], [], [], [], []]
everything = [[], [], [], [], []]

# --- NEW: Lists for magnitude comparison ---
mean_magnitudes = [[], [], [], [], []]
magnitudes_err_up = [[], [], [], [], []]
magnitudes_err_down = [[], [], [], [], []]
everything_mags = [[], [], [], [], []]

mean_diffs = [[], [], [], [], []]
diff_err_up = [[], [], [], [], []]
diff_err_down = [[], [], [], [], []]
everything_diff = [[], [], [], [], []]
# -------------------------------------------

vels_0_target_hat_list = []
vels_0_target_norms_list = [] # NEW: To store the initial magnitudes for comparison

snap_initial = 0
snap_final = 99
snaps = range(snap_initial, snap_final + 1)

np.random.seed(42)

id_list_new = []
vel_list_new = []

print("Computing preliminaries....")

sort_idx_0 = np.argsort(ids_0)
sorted_ids_0 = ids_0[sort_idx_0]

for k in range(5):

    target = id_list[k]

    subset_indices = np.random.choice(len(target), size=9000, replace=False)
    
    ids_0_target = target[subset_indices]

    match_indices_0 = np.searchsorted(sorted_ids_0, ids_0_target)
    original_indices_0 = sort_idx_0[match_indices_0]

    vels_0_target = vels_0[original_indices_0]
    
    vels_0_target_norms = np.linalg.norm(vels_0_target, axis=1)
    
    # Avoid division by zero warnings just in case
    safe_norms = np.where(vels_0_target_norms == 0, 1e-10, vels_0_target_norms)
    vels_0_target_hat = vels_0_target / safe_norms[:, np.newaxis]
    
    vels_0_target_hat_list.append(vels_0_target_hat)
    vels_0_target_norms_list.append(safe_norms) # NEW: Save initial magnitudes

    id_list_new.append(ids_0_target)
    vel_list_new.append(vels_0_target)

for i in snaps:

    if (i % 10 == 0):
        print(f"Working on snaps {i} to {i+9}....")

    ids = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['ParticleIDs'])
    vels = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['Velocities'])

    sort_idx = np.argsort(ids)
    sorted_ids = ids[sort_idx]

    for k in range(5):

        match_indices = np.searchsorted(sorted_ids, id_list_new[k])
        original_indices = sort_idx[match_indices]

        matched_vels = vels[original_indices]
        matched_vels_norms = np.linalg.norm(matched_vels, axis=1)
        
        safe_matched_norms = np.where(matched_vels_norms == 0, 1e-10, matched_vels_norms)
        matched_v_hat = matched_vels / safe_matched_norms[:, np.newaxis]

        # --- ALIGNMENT CALCULATIONS (Original) ---
        dot_products = np.sum(vels_0_target_hat_list[k] * matched_v_hat, axis=1)

        mean_align = np.mean(dot_products)
        higher_align = np.mean(dot_products[np.where(dot_products >= mean_align)]) - mean_align
        lower_align  = mean_align - np.mean(dot_products[np.where(dot_products <= mean_align)]) 

        everything[k].append(dot_products)
        mean_alignments[k].append(mean_align)
        alignments_err_up[k].append(higher_align)
        alignments_err_down[k].append(lower_align)

        # --- MAGNITUDE CALCULATIONS (NEW) ---
        # Calculate the ratio of current magnitude to initial magnitude
        mag_ratios = matched_vels_norms / vels_0_target_norms_list[k]
        mag_diff = matched_vels_norms - vels_0_target_norms_list[k]  

        mean_mag = np.mean(mag_ratios)
        higher_mag = np.mean(mag_ratios[np.where(mag_ratios >= mean_mag)]) - mean_mag
        lower_mag = mean_mag - np.mean(mag_ratios[np.where(mag_ratios <= mean_mag)])

        mean_diff = np.mean(mag_diff)
        higher_diff = np.mean(mag_diff[np.where(mag_diff >= mean_diff)]) - mean_diff
        lower_diff = mean_diff - np.mean(mag_diff[np.where(mag_diff <= mean_diff)])

        everything_mags[k].append(mag_ratios)
        mean_magnitudes[k].append(mean_mag)
        magnitudes_err_up[k].append(higher_mag)
        magnitudes_err_down[k].append(lower_mag)

        mean_diffs[k].append(mean_diff)
        everything_diff[k].append(mag_diff)
        diff_err_up[k].append(higher_diff)
        diff_err_down[k].append(lower_diff)

print("Done!")

In [ ]:
np.save("cos_all.npy", everything)
np.save("cos_avg.npy", mean_alignments)
np.save("cos_err_up.npy", alignments_err_up)
np.save("cos_err_down.npy", alignments_err_down)


np.save("mag_all.npy", everything_mags)
np.save("mag_avg.npy", mean_magnitudes)
np.save("mag_err_up.npy", magnitudes_err_up)
np.save("mag_err_down.npy",magnitudes_err_down)

np.save("diff_all.npy", everything_diff)
np.save("diff_avg.npy", mean_diffs)
np.save("diff_err_up.npy", diff_err_up)
np.save("diff_err_down.npy",diff_err_down)

In [ ]:
red = [ 20.05, 14.99, 11.98, 10.98, 10.00, 9.39, 9.00, 8.45, 8.01, 7.60, 7.24, 7.01, 6.49, 6.01,
    5.85, 5.53, 5.23, 5.00, 4.66, 4.43, 4.18, 4.01, 3.71, 3.49,
    3.28, 3.01, 2.90, 2.73, 2.58, 2.44, 2.32, 2.21, 2.10, 2.00,
    1.90, 1.82, 1.74, 1.67, 1.60, 1.53, 1.50, 1.41, 1.36, 1.30,
    1.25, 1.21, 1.15, 1.11, 1.07, 1.04, 1.00, 0.95, 0.92, 0.89,
    0.85, 0.82, 0.79, 0.76, 0.73, 0.70, 0.68, 0.64, 0.62, 0.60,
    0.58, 0.55, 0.52, 0.50, 0.48, 0.46, 0.44, 0.42, 0.40, 0.38,
    0.36, 0.35, 0.33, 0.31, 0.30, 0.27, 0.26, 0.24, 0.23, 0.21,
    0.20, 0.18, 0.17, 0.15, 0.14, 0.13, 0.11, 0.10, 0.08, 0.07,
    0.06, 0.05, 0.03, 0.02, 0.01, 0.00
]

In [ ]:
everything = np.array(everything)

In [ ]:
np.shape(everything[0][0])

In [ ]:
cos_bins = np.linspace(-1, 1, 10) 
heatmap_data = np.zeros((len(cos_bins) - 1, 100))

for t in range(100):

    hist, _ = np.histogram(everything[0][t], bins=cos_bins)
    heatmap_data[:, t] = hist


fig, ax = plt.subplots(figsize=(8, 5))


im = ax.imshow(
    heatmap_data, 
    extent=[0, 100, cos_bins[0], cos_bins[-1]], 
    origin='lower', 
    aspect='auto', 
    cmap='viridis'
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label(f"Particle Count")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

red_arr=np.array(red)

red_centers = (red_arr[:-1] + red_arr[1:]) / 2
cos_centers = (cos_bins[:-1] + cos_bins[1:]) / 2

# Trim your data matrix to 99 columns if it's currently 100
# (or ensure it's generated as shape (9, 99))
pcm = ax.pcolormesh(
    red_centers, 
    cos_centers, 
    heatmap_data[:, :-1] + 1,  # Trims off the extra 100th column to match the 99 centers
    shading='auto', 
    cmap='viridis', norm = LogNorm()
)

ax.set_xlim(red_arr.max(), red_arr.min())
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(f"Particle Count");

In [ ]:
plt.hist(everything[2][-1], color = "b")
plt.xlabel(r"$\cos \theta$")
plt.ylabel("Counts")
plt.grid();

In [ ]:
ids_0 = il.snapshot.loadSubset(basePath, 0, 'dm', fields = ['ParticleIDs'])
vels_0 = il.snapshot.loadSubset(basePath, 0, 'dm', fields = ['Velocities'])

# Assuming these variables are defined upstream
id_list = [ids_0, node_particle_ids, filament_particle_ids, wall_particle_ids, void_particle_ids]

mean_alignments = [[], [], [], [], []]
alignments_err_up = [[], [], [], [], []]
alignments_err_down = [[], [], [], [], []]
everything = [[], [], [], [], []]

vels_0_target_hat_list = []
id_list_new = []
vel_list_new = []

snap_initial = 0
snap_final = 99
snaps = range(snap_initial, snap_final + 1)

np.random.seed(42)

print("Computing preliminaries....")

# Pre-sort snapshot 0 globally to find initial properties
sort_idx_0 = np.argsort(ids_0)
sorted_ids_0 = ids_0[sort_idx_0]

for k in range(5):
    target = id_list[k]

    # --- FIX 1: Safe Subset Selection ---
    # Ensure you don't try to select 9000 particles if an environment (like voids) has fewer
    sample_size = min(9000, len(target))
    subset_indices = np.random.choice(len(target), size=sample_size, replace=False)
    ids_0_target = target[subset_indices]

    # --- FIX 2: Safe Binary Search Validation ---
    match_indices_0 = np.searchsorted(sorted_ids_0, ids_0_target)
    
    # Clip indices to prevent out-of-bounds errors
    match_indices_0 = np.clip(match_indices_0, 0, len(sorted_ids_0) - 1)
    original_indices_0 = sort_idx_0[match_indices_0]
    
    # Strictly filter out entries where searchsorted did not find an EXACT match
    valid_mask = (ids_0[original_indices_0] == ids_0_target)
    
    # Clean the arrays to only include verified matching particles
    original_indices_0 = original_indices_0[valid_mask]
    ids_0_target = ids_0_target[valid_mask]

    # Fetch and normalize velocities
    vels_0_target = vels_0[original_indices_0]
    vels_0_target_norms = np.linalg.norm(vels_0_target, axis=1)
    
    # Prevent division by zero if a particle is completely stationary
    vels_0_target_norms[vels_0_target_norms == 0] = 1.0
    vels_0_target_hat = vels_0_target / vels_0_target_norms[:, np.newaxis]
    
    vels_0_target_hat_list.append(vels_0_target_hat)
    id_list_new.append(ids_0_target)
    vel_list_new.append(vels_0_target)

# --- Snapshot Tracking Loop ---
for i in snaps:
    if (i % 10 == 0):
        print(f"Working on snaps {i} to {i+9}....")

    ids = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['ParticleIDs'])
    vels = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['Velocities'])

    sort_idx = np.argsort(ids)
    sorted_ids = ids[sort_idx]

    for k in range(5):
        # Match current snapshot against your strictly tracked baseline subset
        match_indices = np.searchsorted(sorted_ids, id_list_new[k])
        match_indices = np.clip(match_indices, 0, len(sorted_ids) - 1)
        original_indices = sort_idx[match_indices]

        # --- FIX 3: Dynamic Snapshot Validation ---
        # Particles can occasionally leave the simulation boundary or sublink tracking. 
        # Check that the snapshot particle matches your tracking ID.
        valid_mask = (ids[original_indices] == id_list_new[k])
        
        matched_vels = vels[original_indices]
        matched_vels_norms = np.linalg.norm(matched_vels, axis=1)
        matched_vels_norms[matched_vels_norms == 0] = 1.0
        matched_v_hat = matched_vels / matched_vels_norms[:, np.newaxis]

        # Use the valid mask to calculate dot products only for confirmed matching particle pairs
        v0_hat = vels_0_target_hat_list[k][valid_mask]
        vt_hat = matched_v_hat[valid_mask]

        dot_products = np.sum(v0_hat * vt_hat, axis=1)

        # Handle edge case if no particles matched
        if len(dot_products) == 0:
            mean, higher, lower = np.nan, np.nan, np.nan
        else:
            mean = np.mean(dot_products)
            higher_pts = dot_products[dot_products >= mean]
            lower_pts = dot_products[dot_products <= mean]
            
            higher = np.mean(higher_pts) - mean if len(higher_pts) > 0 else 0
            lower  = mean - np.mean(lower_pts) if len(lower_pts) > 0 else 0

        everything[k].append(dot_products)
        mean_alignments[k].append(mean)
        alignments_err_up[k].append(higher)
        alignments_err_down[k].append(lower)

print("Done!")

In [ ]:
struct = ["All","Node","Filament","Wall","Void"]

fig = plt.figure(figsize=(10,4))
ax = fig.add_subplot(121)

plt.style.use('seaborn-v0_8-colorblind')

styles = ["solid","-.","--",":"]

ax.plot(red, mean_alignments[0], label = f"{struct[0]}", c="black")
ax.fill_between(red, (np.array(mean_alignments[0])-np.array(alignments_err_down[0])), 
                (np.array(mean_alignments[0])+np.array(alignments_err_up[0])), alpha=.2, color="black")
    
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"cos$\theta$")
ax.set_title("Mean alignment to the initial velocity")
ax.legend()
ax.grid()
#ax.tick_params(direction='in')

ax = fig.add_subplot(122)

for k in range(1,5):
    ax.plot(red[25:], mean_alignments[k][25:], label = f"{struct[k]}", linestyle = styles[k-1])

        
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"cos$\theta$")
ax.set_title("Mean alignment to the initial velocity")
ax.legend()
ax.grid()

plt.tight_layout();

In [ ]:
ids_0 = il.snapshot.loadSubset(basePath, 0, 'dm', fields = ['ParticleIDs'])
vels_0 = il.snapshot.loadSubset(basePath, 0, 'dm', fields = ['Velocities'])

id_list = [ids_0, node_particle_ids, filament_particle_ids, wall_particle_ids, void_particle_ids]

mean_alignments = [[], [], [], [], []]
alignments_err_up = [[], [], [], [], []]
alignments_err_down = [[], [], [], [], []]

vels_0_target_hat_list = []

snap_initial = 0
snap_final = 99
snaps = range(snap_initial, snap_final + 1)

np.random.seed(42)

id_list_new = []
vel_list_new = []

print("Computing preliminaries....")

sort_idx_0 = np.argsort(ids_0)
sorted_ids_0 = ids_0[sort_idx_0]

for k in range(5):

    target = id_list[k]

    subset_indices = np.random.choice(len(target), size=9000, replace=False)
    
    ids_0_target = target[subset_indices]

    match_indices_0 = np.searchsorted(sorted_ids_0, ids_0_target)
    original_indices_0 = sort_idx_0[match_indices_0]

    vels_0_target = vels_0[original_indices_0]
    
    vels_0_target_norms = np.linalg.norm(vels_0_target, axis=1)
    vels_0_target_hat = vels_0_target / vels_0_target_norms[:, np.newaxis]
    
    vels_0_target_hat_list.append(vels_0_target_hat)

    id_list_new.append(ids_0_target)
    vel_list_new.append(vels_0_target)

snapss = [1,25,50,75,99]

dots = []

for i in snapss:

    if (i % 10 == 0):
        print(f"Working on snaps {i} to {i+9 }....")

    ids = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['ParticleIDs'])
    vels = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['Velocities'])

    sort_idx = np.argsort(ids)
    sorted_ids = ids[sort_idx]

    for k in range(5):

        match_indices = np.searchsorted(sorted_ids, id_list_new[k])
        original_indices = sort_idx[match_indices]

        matched_vels = vels[original_indices]
        matched_vels_norms = np.linalg.norm(matched_vels, axis=1)
        matched_v_hat = matched_vels / matched_vels_norms[:, np.newaxis]

        dot_products = np.sum(vels_0_target_hat_list[k] * matched_v_hat, axis=1)

        mean = np.mean(dot_products)
        higher = np.mean(dot_products[np.where(dot_products >= mean)]) - mean
        lower  = mean - np.mean(dot_products[np.where(dot_products <= mean)]) 
    
        mean_alignments[k].append(mean)
        alignments_err_up[k].append(higher)
        alignments_err_down[k].append(lower)

        if(k==1):
            dots.append(dot_products)

print("Done!")


In [ ]:
plt.hist(dots[-1], color = "b")
plt.xlabel(r"$\cos \theta$")
plt.ylabel("Counts")
plt.grid();

In [ ]:
print(mean_alignments[0][4])

In [ ]:
print(alignments_err_down[0][4])

In [ ]:
snap_initial = 0
snap_final = 99

ids_0 = il.snapshot.loadSubset(basePath, snap_initial, 'dm', fields = ['ParticleIDs'])
vels_0 = il.snapshot.loadSubset(basePath, snap_initial, 'dm', fields = ['Velocities'])

np.random.seed(42)
subset_indices = np.random.choice(len(ids_0), size=10000, replace=False)

ids_0_target = ids_0[subset_indices]
vels_0_target = vels_0[subset_indices]

vels_0_target_norms = np.linalg.norm(vels_0_target, axis=1)
vels_0_target_hat = vels_0_target / vels_0_target_norms[:, np.newaxis]

snaps = range(snap_initial, snap_final + 1)
mean_alignments = []

previous = vels_0_target_hat

for i in snaps:
    if (i % 10 == 0):
        print(f"Working on snaps {i} to {i+9 }....")

    part = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['ParticleIDs','Velocities' ])
    ids = part['ParticleIDs']
    vels = part['Velocities']

    sort_idx = np.argsort(ids)
    sorted_ids = ids[sort_idx]
    
    match_indices = np.searchsorted(sorted_ids, ids_0_target)
    original_indices = sort_idx[match_indices]
    
    matched_vels = vels[original_indices]
    
    matched_vels_norms = np.linalg.norm(matched_vels, axis=1)
    matched_v_hat = matched_vels / matched_vels_norms[:, np.newaxis]
    
    dot_products = np.sum(previous * matched_v_hat, axis=1)

    previous = matched_v_hat
    
    mean_alignments.append(np.mean(dot_products))

print("Done!")


In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
ax.plot(red, mean_alignments, c="b")
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"cos$\theta$")
ax.set_title("Mean alignment to the previous velocity")
#ax.axhline(0, color='k', linestyle='--')
ax.grid();

In [ ]:
np.shape(everything[0])